# Stage LR — Reranker Diagnostic

**This is a diagnostic, not a deployment candidate.** Per direct instruction: even a good held-out result from this notebook does **not** clear Gate 2 (a substantially larger, independently collected, more diverse dataset than what's been used so far — still an open, unset parameter, per `VALIDATION.md` §56's reopening condition (1) and this project's own `LEARNED_REFORMULATION_RESEARCH.md` charter). What this notebook actually tests: **is the reranker approach itself viable at all, once the specific training mistakes from the last attempt are fixed** — nothing more, nothing less. The held-out numbers get reported plainly at the end regardless of whether they look good or bad.

**Why a reranker, not a better generator.** This project's frozen rule-based pipeline (`architecture-freeze-v1`) plateaued with its dominant remaining defect class (`WRONG_WORD_OR_SENSE`) diagnosed as structurally a *ranking* problem, not a generation problem — candidate rewordings are usually available, the fixed `combined_score()` heuristic (90% semantic similarity / 10% frequency) just doesn't reliably pick the best one. A learned reranker is the natural response to that specific diagnosis.

**What went wrong last time, precisely (not vague caution — the actual diagnosis), and what's fixed here:**

| Phase 9's problem | Root cause found | Fix applied in this notebook |
|---|---|---|
| Training diverged to NaN (Phase 9) | `pos_weight≈11` (aggressive class-imbalance correction) + `lr=2e-5`, a known gradient-explosion recipe on a 205-example train set | `pos_weight` not needed here — this task's A/B preference pairs are naturally close to balanced, unlike CLEAN/DEFECTIVE's 17:188 skew |
| Fixed (Phase 9B/9C) but didn't generalize: predicted DEFECTIVE 99% of the time on a genuinely fresh, disjoint corpus (`VALIDATION.md` §46) | Training data was small (205 examples) **and not split by speaker/profile** — the model could look good on its own held-out rows while still overfitting to the specific templates/profiles those rows came from | **Held-out-by-speaker split, from the very first cell** — no profile/participant appears in more than one of train/val/test |
| A first eval pass used a coarse threshold grid and looked like no signal existed | Fine-grained threshold sweep needed | N/A here — this is a binary A/B classification with a fixed 0.5 decision boundary, no threshold search |

**Honest disclosure about this notebook itself:** it was written and reviewed carefully, but not executed — there is no GPU/Colab access in the environment that wrote it. Expect to debug on the first real run, same as any new notebook. Report back what actually happens rather than assuming it works as written.

## 0. Setup

In [ ]:
!pip install -q transformers accelerate scikit-learn torch --upgrade

In [ ]:
import json
import random
from pathlib import Path

import numpy as np
import torch

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
print("device:", DEVICE)
if DEVICE == "cpu":
    print("WARNING: no GPU detected — Runtime > Change runtime type > GPU, then re-run.")

## 1. Load data — privacy note

Three possible sources, all optional except the first:

1. **`lr1_preference_pairs.json`** — 75 synthetic-template pairs, tracked in the project's own git repo (`stage_lr/data/lr1_preference_pairs.json` on the `stage-lr` branch). No privacy concern — researcher-authored templates, not real people's data. Safe to pull directly from GitHub.
2. **`real_human_pairs.json`** — real participants' declared profiles and judgments. This file is **gitignored and never leaves local machines** by this project's own standing rule. It is **not** in the GitHub repo. If you want to include it here, you must upload it yourself, from your own local copy, as a deliberate action each time — it is never fetched automatically.
3. **`claude_only_pairs.json`** — same rule as above: real participants' declared words, gitignored, upload only if you choose to.

Skip the upload cell entirely to train on synthetic-only data (source 1 only) — the notebook works either way, it just reports fewer held-out groups without 2/3.

In [ ]:
# Source 1 — synthetic pairs, safe to pull directly (public, tracked, no participant content).
!wget -q -O lr1_preference_pairs.json https://raw.githubusercontent.com/haqiqak/speech-ai/stage-lr/stage_lr/data/lr1_preference_pairs.json
print("downloaded lr1_preference_pairs.json:", Path("lr1_preference_pairs.json").stat().st_size, "bytes")

In [ ]:
# Sources 2 and 3 — OPTIONAL, real participant data. Only run this cell if you
# choose to include it, uploading from your own local machine each time.
# Skip this cell entirely to train on synthetic-only data.
try:
    from google.colab import files
    print("Upload real_human_pairs.json and/or claude_only_pairs.json now (or press Cancel to skip):")
    uploaded = files.upload()
    for name in uploaded:
        print("received:", name)
except ImportError:
    print("Not running in Colab — place real_human_pairs.json / claude_only_pairs.json in the working directory manually if you want them included.")

## 2. Normalize all sources into one schema

In [ ]:
def build_sentence(original_sentence: str, flagged_word: str, candidate: str) -> str:
    """Best-effort reconstruction: swap the candidate word into the original
    sentence in place of the flagged word. Good enough for a cross-encoder
    input -- exact surface form (inflection) doesn't need to be perfect for
    the model to learn a preference signal, unlike the human-facing relay
    text used elsewhere in this project."""
    import re
    pattern = re.compile(re.escape(flagged_word), re.IGNORECASE)
    if pattern.search(original_sentence):
        return pattern.sub(candidate, original_sentence, count=1)
    return f"{original_sentence} [{candidate}]"  # fallback, should rarely fire


def normalize_synthetic(path: str) -> list[dict]:
    if not Path(path).exists():
        return []
    data = json.loads(Path(path).read_text(encoding="utf-8"))
    rows = []
    for p in data["pairs"]:
        if p.get("preferred") not in ("A", "B"):
            continue  # drop ties and anything unlabeled
        profile = p.get("difficulty_profile", {})
        group_key = "synthetic:" + (profile.get("name") or json.dumps(profile, sort_keys=True))
        rows.append({
            "sentence_with_A": build_sentence(p["original_sentence"], p["flagged_word"], p["candidate_A"]),
            "sentence_with_B": build_sentence(p["original_sentence"], p["flagged_word"], p["candidate_B"]),
            "label": 1 if p["preferred"] == "A" else 0,
            "group_key": group_key,
            "source": "synthetic_profile_template",
        })
    return rows


def normalize_real_human(path: str) -> list[dict]:
    """real_human_pairs.json -- uses the HUMAN verdict as the training
    label (this is the gold signal we actually want a reranker to learn),
    not the Claude verdict, which is only a comparison point elsewhere."""
    if not Path(path).exists():
        return []
    data = json.loads(Path(path).read_text(encoding="utf-8"))
    rows = []
    for p in data["pairs"]:
        if p.get("human_preferred") not in ("A", "B"):
            continue
        rows.append({
            "sentence_with_A": build_sentence(p["original_sentence"], p["flagged_word"], p["candidate_A"]),
            "sentence_with_B": build_sentence(p["original_sentence"], p["flagged_word"], p["candidate_B"]),
            "label": 1 if p["human_preferred"] == "A" else 0,
            "group_key": "participant:" + p["participant_label"],
            "source": "real_human",
        })
    return rows


def normalize_claude_only(path: str) -> list[dict]:
    if not Path(path).exists():
        return []
    data = json.loads(Path(path).read_text(encoding="utf-8"))
    rows = []
    for p in data["pairs"]:
        if p.get("claude_preferred") not in ("A", "B"):
            continue
        rows.append({
            "sentence_with_A": build_sentence(p["original_sentence"], p["flagged_word"], p["candidate_A"]),
            "sentence_with_B": build_sentence(p["original_sentence"], p["flagged_word"], p["candidate_B"]),
            "label": 1 if p["claude_preferred"] == "A" else 0,
            "group_key": "participant:" + p["participant_label"],
            "source": "real_profile_claude_only",
        })
    return rows


all_rows = (
    normalize_synthetic("lr1_preference_pairs.json")
    + normalize_real_human("real_human_pairs.json")
    + normalize_claude_only("claude_only_pairs.json")
)

print(f"total labeled rows: {len(all_rows)}")
from collections import Counter
print("by source:", Counter(r["source"] for r in all_rows))
print("distinct groups (speakers/templates):", len(set(r["group_key"] for r in all_rows)))
if len(all_rows) < 100:
    print("\nNOTE: this is a small dataset. A weak or noisy held-out result is expected "
          "and informative on its own terms -- it does not mean the method is broken, "
          "per this notebook's own framing above.")

## 3. Held-out-by-speaker split

No `group_key` (a speaker/profile identity) may appear in more than one split — this is the specific fix for the failure mode found in `VALIDATION.md` §46 (a model that looked fine on its own held-out *rows* still collapsed on genuinely new *profiles*).

In [ ]:
groups = sorted(set(r["group_key"] for r in all_rows))
random.Random(SEED).shuffle(groups)

n_groups = len(groups)
if n_groups < 3:
    raise SystemExit(
        f"Only {n_groups} distinct group(s) available -- cannot form a meaningful "
        "held-out-by-speaker split (need at least 1 group per split, ideally several). "
        "This is itself a real, reportable result: the data isn't diverse enough yet "
        "to run this diagnostic meaningfully. Upload real_human_pairs.json / "
        "claude_only_pairs.json above, or collect more distinct profiles, then re-run."
    )

# Rough 70/15/15 split by GROUP count, not row count -- a group with many rows
# should not let one split dominate.
n_test = max(1, round(n_groups * 0.15))
n_val = max(1, round(n_groups * 0.15))
test_groups = set(groups[:n_test])
val_groups = set(groups[n_test:n_test + n_val])
train_groups = set(groups[n_test + n_val:])

assert not (train_groups & val_groups) and not (train_groups & test_groups) and not (val_groups & test_groups)

train_rows = [r for r in all_rows if r["group_key"] in train_groups]
val_rows = [r for r in all_rows if r["group_key"] in val_groups]
test_rows = [r for r in all_rows if r["group_key"] in test_groups]

print(f"train: {len(train_rows)} rows / {len(train_groups)} groups")
print(f"val:   {len(val_rows)} rows / {len(val_groups)} groups")
print(f"test:  {len(test_rows)} rows / {len(test_groups)} groups")
print("\ntest groups (should never appear in train/val):", sorted(test_groups))

if len(test_rows) < 10:
    print("\nNOTE: test split is very small -- treat the held-out number as directional "
          "only, not a precise estimate. Report the raw count alongside the rate, always.")

## 4. Model, data collator, safety callback

In [ ]:
from transformers import (
    AutoTokenizer, AutoModelForSequenceClassification,
    Trainer, TrainingArguments, TrainerCallback,
)

MODEL_NAME = "microsoft/deberta-v3-xsmall"  # same family/size class as Phase 9's own attempt

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
model = AutoModelForSequenceClassification.from_pretrained(MODEL_NAME, num_labels=2)


class HaltOnNonFinite(TrainerCallback):
    """Direct port of Phase 9B's own fix (VALIDATION.md SS44): abort the
    instant any weight goes non-finite, rather than burning through dead
    epochs the way Phase 9's original run did."""
    def on_step_end(self, args, state, control, model=None, **kwargs):
        for p in model.parameters():
            if not torch.isfinite(p).all():
                print(f"NON-FINITE WEIGHT at step {state.global_step} -- aborting training.")
                control.should_training_stop = True
                break
        return control


def tokenize_rows(rows):
    enc = tokenizer(
        [r["sentence_with_A"] for r in rows],
        [r["sentence_with_B"] for r in rows],
        truncation=True, padding="max_length", max_length=96,
    )
    enc["labels"] = [r["label"] for r in rows]
    return enc


class PairDataset(torch.utils.data.Dataset):
    def __init__(self, encodings):
        self.encodings = encodings
    def __len__(self):
        return len(self.encodings["labels"])
    def __getitem__(self, idx):
        return {k: torch.tensor(v[idx]) for k, v in self.encodings.items()}


train_ds = PairDataset(tokenize_rows(train_rows))
val_ds = PairDataset(tokenize_rows(val_rows))
test_ds = PairDataset(tokenize_rows(test_rows))

## 5. Train — conservative recipe, per Phase 9B's diagnosed fixes

In [ ]:
def compute_metrics(eval_pred):
    logits, labels = eval_pred
    preds = np.argmax(logits, axis=-1)
    acc = (preds == labels).mean()
    # class balance check -- did the model just learn to always predict one class?
    pred_a_rate = (preds == 1).mean()
    return {"accuracy": float(acc), "pred_A_rate": float(pred_a_rate)}


training_args = TrainingArguments(
    output_dir="./reranker_diagnostic_out",
    num_train_epochs=8,
    learning_rate=3e-6,          # Phase 9B's conservative LR, not Phase 9's failed 2e-5
    per_device_train_batch_size=8,
    per_device_eval_batch_size=16,
    max_grad_norm=1.0,           # explicit clipping, confirmed already-active-but-insufficient
                                  # in Phase 9 alone -- kept anyway, cheap insurance
    adam_epsilon=1e-6,           # vs. library default 1e-8, part of Phase 9B's fix
    eval_strategy="epoch",
    save_strategy="epoch",
    load_best_model_at_end=True,
    metric_for_best_model="eval_loss",
    greater_is_better=False,
    save_total_limit=2,
    seed=SEED,
    report_to="none",
)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_ds,
    eval_dataset=val_ds,
    compute_metrics=compute_metrics,
    callbacks=[HaltOnNonFinite()],
)

train_result = trainer.train()
print(train_result)

## 6. Held-out evaluation — report plainly, whatever it says

In [ ]:
test_metrics = trainer.evaluate(test_ds, metric_key_prefix="test")
print(json.dumps(test_metrics, indent=2))

# Sanity baselines, computed directly, not assumed:
majority_label = 1 if sum(r["label"] for r in train_rows) >= len(train_rows) / 2 else 0
majority_baseline_acc = sum(1 for r in test_rows if r["label"] == majority_label) / max(1, len(test_rows))
print(f"\nchance baseline: 0.500")
print(f"majority-class baseline (from train split): {majority_baseline_acc:.3f}")
print(f"model held-out accuracy: {test_metrics.get('test_accuracy', float('nan')):.3f}")
print(f"model's rate of predicting A on held-out: {test_metrics.get('test_pred_A_rate', float('nan')):.3f} "
      f"(compare to actual A-rate: {sum(r['label'] for r in test_rows) / max(1, len(test_rows)):.3f} "
      "-- if these are very different, the model likely collapsed to one class, "
      "the exact Phase 9C failure mode)")

## 7. What this result does and does not mean

**Read the numbers above against these two questions, not against "is this good enough to ship":**

1. **Does held-out accuracy clear chance (50%) and the majority-class baseline, without collapsing to always predicting one class?** If yes: the reranker approach itself shows a real, non-degenerate signal once Phase 9's specific training mistakes are fixed — worth continuing to invest in as data grows. If no (including a Phase-9C-style collapse to one class): that's also a real, informative result — it means the fixes applied here weren't sufficient on their own, and the small/narrow dataset is likely still the binding constraint, not the training recipe.
2. **Either way, this does not clear Gate 2.** `VALIDATION.md` §56's reopening condition (1) requires "a substantially larger, independently collected labeled dataset" — this run's held-out set has only as many distinct groups as were uploaded above, almost certainly far short of that bar. A good number here is a green light to keep collecting data toward that bar, not a green light to deploy anything.

Save and report the numbers from section 6 exactly as printed, including if they're bad — that's the actual point of running this.

In [ ]:
summary = {
    "n_total_rows": len(all_rows),
    "n_groups_total": n_groups,
    "train": {"rows": len(train_rows), "groups": len(train_groups)},
    "val": {"rows": len(val_rows), "groups": len(val_groups)},
    "test": {"rows": len(test_rows), "groups": len(test_groups)},
    "chance_baseline": 0.5,
    "majority_class_baseline": majority_baseline_acc,
    "test_metrics": test_metrics,
}
Path("reranker_diagnostic_results.json").write_text(json.dumps(summary, indent=2))
print(json.dumps(summary, indent=2))

try:
    from google.colab import files
    files.download("reranker_diagnostic_results.json")
except ImportError:
    print("\nSaved to reranker_diagnostic_results.json -- download it manually if not in Colab.")